# Distance Stats (_`R` interpreter_)

In [ ]:
library(tidyverse)
theme_set(theme_classic())

## Aes

In [ ]:
labs_map <- c(
  "Adipocyte"   = "Adipocyte",
  "Bcell"       = "Bcell",
  "CD4_CCR6"    = "CD4 Tcell CCR6+",
  "CD4_GZMB"    = "CD4 Tcell GZMB+",
  "CD4_Other"   = "CD4 Tcell other",
  "CD4_Tex"     = "CD4 Tex",
  "CD4_Tmem"    = "CD4 Tmem",
  "CD4_Tn"      = "CD4 Tnaive",
  "CD4_Treg"    = "CD4 Treg",
  "CD8_GZMB"    = "CD8 Tcell GZMB+",
  "CD8_naive"   = "CD8 Tnaive",
  "CD8_Other"   = "CD8 Tcell other",
  "CD8_Tex"     = "CD8 Tex",
  "CD8_Tmem"    = "CD8 Tmem",
  "DC"          = "DC",
  "EC"          = "EC",
  "Lin-"        = "Unclassified",
  "Mac_M1"      = "M2-like macrophage",
  "Mac_M2"      = "M1-like macrophage",
  "MKs"         = "MK",
  "Mono_CD14"   = "Monocytes CD14+",
  "Mono_CD16"   = "Monocytes CD16+",
  "Mye_HLADR"   = "Myeloid HLA-DR+",
  "Mye_other"   = "Myeloid other",
  "Myeloma"     = "Myeloma",
  "NK"          = "NK cells"
)

## Funs

In [ ]:
perCN_comp = function(bl_distances, CN_, dist=200){

  density_CN_ <- bl_distances |>
    filter(labels == "Myeloma") |>
    pivot_longer(cols = all_of(distance_lables), names_to = "cell_partner", values_to = "cell_distance") |>
    filter(cell_distance < dist) %>% filter(cn_celltypes==CN_)
  
  density_Other <- bl_distances |>
    filter(labels == "Myeloma") |>
    pivot_longer(cols = all_of(distance_lables), names_to = "cell_partner", values_to = "cell_distance") |>
    filter(cell_distance < dist) %>% filter(cn_celltypes!=CN_)
  
  cell_out=c()
  p_out=c()
  log2fc_out=c()
  for (cell_test in unique(density_CN_$cell_partner)){
    d1=density_CN_[density_CN_$cell_partner==cell_test,]$cell_distance
    d2=density_Other[density_Other$cell_partner==cell_test,]$cell_distance
    if (length(d1)<10|length(d2)<10){next}
    cell_out=c(cell_out,cell_test)
    p_=wilcox.test(d1,d2)$p.value
    p_out=c(p_out,p_)
    log2fc_out=c(log2fc_out, log2( median(d1)/median(d2)))
  }
  
  res=data.frame(cell=cell_out, p=p_out, log2fc=log2fc_out) %>% mutate(fdr=p.adjust(p)) %>% arrange(p)
  
  return(res)
  
  }

## Baseline CN

### Import

In [ ]:
bl_distances = read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/bslTP_linNegOut_15clus-distances.csv')
bl_distances %>% head(1)

In [ ]:
distance_lables <- bl_distances |> select("Adipocyte":"NK") |> colnames() 

perCN_comp_results = list()

for ( CN in unique(bl_distances$cn_celltypes) ){
  perCN_comp_results[[CN]]=perCN_comp(bl_distances, CN, dist=200)
}

res = bind_rows(perCN_comp_results, .id='comparison') %>% as_tibble() %>% 
  mutate(cell = factor(labs_map[cell],rev(as.vector(labs_map))))

In [ ]:
res %>% write.csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/dist_stats-bslTP_linNegOut_15clus.csv', row.names=F)
res = read_csv('/mnt/disks/data/imc/CART_cohort/out/v1_pheno_work/cn_data/dist_stats-bslTP_linNegOut_15clus.csv')

In [ ]:
res %>% 
  mutate(CN=comparison) %>% 
  filter(CN %in% c('CN13','CN14','CN2')) %>%
  mutate(
      change = ifelse(fdr>0.05,'NS',
                      ifelse(abs(log2fc)<1,'FDR < 0.05',
                             ifelse(log2fc>1,'FDR < 0.05\nlog2FC > 1\n','FDR < 0.05\nlog2FC < 1\n'
                                   )
                            )
                     )
  ) %>%
  mutate(cell=factor(cell,levels=rev(as.vector(labs_map)))) %>% 
  mutate(change=factor(change,
      levels=c('FDR < 0.05\nlog2FC > 1\n','FDR < 0.05\nlog2FC < 1\n','FDR < 0.05','NS')
  )) %>%

  ggplot(aes(log2fc,cell,fill=change))+
  geom_vline(xintercept=0)+geom_vline(xintercept = c(-1,1),color='dodgerblue',linetype='dashed')+
  geom_point(shape=21,size=3)+
  labs(x='Fold-change CN vs other (log2)',y='',fill='',title='bslTP_linNegOut_15clus')+
  facet_wrap('CN',ncol=8) + 
  scale_x_continuous()+
  scale_fill_manual(values=c('FDR < 0.05\nlog2FC > 1\n'='blue',
                             'FDR < 0.05\nlog2FC < 1\n'='maroon',
                             'FDR < 0.05'='black',
                             'NS'='grey70'))+
  xlim(c(-3,3))+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.title = element_blank(),
    axis.line=element_line(),panel.grid.major = element_line(color = "grey80"),
    panel.grid.major.x = element_blank()
  )+theme_bw()-> pl
pl
ggsave("/mnt/disks/data/imc/CART_cohort/out/final_figures/Fig_5G.pdf", pl,height=7,width=7,dpi=300)

In [ ]:
res %>% 
  mutate(CN=comparison) %>% 
  filter(! CN %in% c('CN13','CN14','CN2')) %>%
  mutate(
      change = ifelse(fdr>0.05,'NS',
                      ifelse(abs(log2fc)<1,'FDR < 0.05',
                             ifelse(log2fc>1,'FDR < 0.05\nlog2FC > 1\n','FDR < 0.05\nlog2FC < 1\n'
                                   )
                            )
                     )
  ) %>%
  mutate(cell=factor(cell,levels=rev(as.vector(labs_map)))) %>% 
  mutate(change=factor(change,
      levels=c('FDR < 0.05\nlog2FC > 1\n','FDR < 0.05\nlog2FC < 1\n','FDR < 0.05','NS')
  )) %>%

  ggplot(aes(log2fc,cell,fill=change))+
  geom_vline(xintercept=0)+geom_vline(xintercept = c(-1,1),color='dodgerblue',linetype='dashed')+
  geom_point(shape=21,size=3)+
  labs(x='Fold-change CN vs other (log2)',y='',fill='',title='bslTP_linNegOut_15clus')+
  facet_grid(~CN) + 
  scale_x_continuous()+
  scale_fill_manual(values=c('FDR < 0.05\nlog2FC > 1\n'='blue',
                             'FDR < 0.05\nlog2FC < 1\n'='maroon',
                             'FDR < 0.05'='black',
                             'NS'='grey70'))+
  xlim(c(-3,3))+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.title = element_blank(),
    axis.line=element_line(),panel.grid.major = element_line(color = "grey80"),
    panel.grid.major.x = element_blank()
  )+theme_bw()-> pl
pl
ggsave("/mnt/disks/data/imc/CART_cohort/out/final_figures/Fig_S5F.pdf", pl,height=7,width=17,dpi=300)

.PDF conversion
```
sudo apt-get install pdf2svg
pdf2svg  out/final_figures/Fig_5G.pdf  out/final_figures/Fig_5G.svg
pdf2svg  out/final_figures/Fig_S5F.pdf  out/final_figures/Fig_S5F.svg
```